# Labelling time: one number to quote

Person-time per label from Label Studio's `lead_time`, for annotations drawn **from scratch** (no prediction
or earlier label pre-filled), across all four label types (laser dot, snout + fork, species, slate corners).
Source: `data/labeling/lead_times.csv` (restored production backup; no annotator identities).

`lead_time` counts the time a task was open, so an annotator who leaves a tab open adds hours to one label.
The raw mean is therefore dominated by idle tabs and is not quoted; the median and a 10%-trimmed mean are.

In [1]:
import pandas as pd, numpy as np
from scipy import stats
from fishsense_cscw.paths import DATA

d = pd.read_csv(DATA / "labeling" / "lead_times.csv")
scratch = d[d.seeded.astype(str).str.lower().isin(["f", "false"])]
s = scratch.lead_time_s
print(f"{len(s):,} labels drawn from scratch")

78,977 labels drawn from scratch


In [2]:
summary = pd.DataFrame({
    "labels": [len(s)],
    "median_s": [s.median()],
    "trimmed_mean_10pct_s": [stats.trim_mean(s, 0.1)],
    "mean_excl_over_1h_s": [s[s <= 3600].mean()],
    "raw_mean_s": [s.mean()],
    "share_over_1h": [(s > 3600).mean()],
}).round(3)
summary.T

,0
labels,78977.000
median_s,13.950
trimmed_mean_10pct_s,15.587
mean_excl_over_1h_s,31.738
raw_mean_s,137.150
share_over_1h,0.003


In [3]:
by_type = scratch.groupby("label_type").lead_time_s.agg(
    labels="size", median_s="median", trimmed_mean_10pct_s=lambda v: stats.trim_mean(v, 0.1)).round(1)
by_type

,labels,median_s,trimmed_mean_10pct_s
label_type,,,
head_tail,41435,15.0,17.0
laser_dot,32373,12.9,13.8
slate_corners,483,61.3,67.5
species,4686,12.2,14.6


In [4]:
per_frame = by_type.loc[["laser_dot", "head_tail", "species"], "median_s"].sum()
print(f"QUOTE: a person spends a median of {s.median():.0f} s per label "
      f"(10%-trimmed mean {stats.trim_mean(s, 0.1):.1f} s, n = {len(s):,}); "
      f"a measured frame (dot + snout/fork + species) takes about {per_frame:.0f} s.")

QUOTE: a person spends a median of 14 s per label (10%-trimmed mean 15.6 s, n = 78,977); a measured frame (dot + snout/fork + species) takes about 40 s.
